## Key Concepts of CrewAI:

- Agents: Autonomous entities within crewAI capable of executing tasks independently or collaboratively.

- Tasks: Defined objectives, goals or actions assigned to agents for execution.

- Tools: Integrated functionalities or APIs that agents can utilize to accomplish tasks.

- Processes: Sequential or hierarchical workflows that guide task execution.

- Crews: Groups of agents working together towards a common goal.

- Training: Continuous improvement and learning mechanisms for agents.

- Memory: Persistent storage for learned information and context retention. 

### References
- https://docs.crewai.com/introduction
- https://python.langchain.com/docs/integrations/tools/ddg/

### Load environment variables

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Initialize our LLM 

In [3]:
from crewai import LLM

llm = LLM(
    model = "groq/llama-3.3-70b-versatile",
    temperature = 0.7
)

### Define the Tools

- Integration with LangChain Tools

In [4]:
from crewai_tools import tool, BaseTool
from pydantic import Field
from langchain_community.tools import DuckDuckGoSearchRun

In [5]:
class SearchTool(BaseTool):
    name: str = "Search"
    description: str = "Useful for search-based queries. Use this to find current information about markets, companies, and trends."
    search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)

    def _run(self, query: str) -> str:
        """Execute the search query and return results"""
        try:
            return self.search.invoke(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"


### Create Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

In [6]:
from crewai import Agent

researcher = Agent(
    role = "Market Research Analyst",
    goal = "Provide up-to-date market analysis of the {industry} industry",
    backstory = "An expert analyst with a keen eye for industry trends and market data.",
    verbose = True,
    allow_delegation = False,
    llm = llm,
    tools = [SearchTool()]
)

writer = Agent(
    role = "Content Writer",
    goal = "Craft engaging blog posts about the {industry} sector based on provided research.",
    backstory = "A creative professional skilled in delivering impactful content.",
    verbose = True,
    allow_delegation = False,
    llm = llm,
)

### Define Tasks

In [8]:
from crewai import Task

research = Task(
    description = 'Research the latest trends in the {industry} industry and provide a summary.',
    expected_output = 'A summary of the top 2 trending developments in the {industry} industry with a unique perspective on their significance.',
    agent = researcher
)

write = Task(
    description = "Write an engaging blog post about the AI industry, based on the research analyst's summary.",
    expected_output = "A well-written blog post in markdown format, ready for publication, each section should have 1 paragraph.",
    agent = writer,
    output_file = 'new_post.md'  
)

### Create a crew of agents and execute 

- Pass the tasks to be performed by those agents.
- Note: For this example, the tasks will be performed sequentially (i.e they are dependent on each other), so the order of the task in the list matters.

In [9]:
from crewai import Crew, Process 

crew = Crew(
    agents = [researcher, writer],
    tasks = [research, write],
    process = Process.sequential,
    verbose = True
)

In [10]:
result = crew.kickoff(inputs = {"industry": "Artificial Intelligence"})

# Agent: Market Research Analyst
## Task: Research the latest trends in the Artificial Intelligence industry and provide a summary.


# Agent: Market Research Analyst
## Thought: Thought: To provide a comprehensive summary of the latest trends in the Artificial Intelligence industry, I need to gather information on the current developments and advancements in this field. I will utilize the Search tool to find relevant data and insights.
## Using tool: Search
## Tool Input: 
"{\"query\": \"latest trends in Artificial Intelligence\"}"
## Tool Output: 
Discover the 10 major AI trends set to reshape 2025: from augmented working and real-time decision-making to advanced AI legislation and sustainable AI initiatives. Learn about the latest developments and trends in AI, such as multimodal, agentic and open source AI, and how they will transform the industry. This article covers the challenges, opportunities and applications of AI in various domains, from healthcare to finance. Learn how gene